# Assignment 27 — Chatbots with Conversation History using LangChain

This notebook follows Tasks 1–7 of the assignment. The model is Ollama through LangChain.
Execution outputs should be generated locally; no fake model responses are included.

## Task 1 — Message Types

**SystemMessage:** gives the assistant instructions about its role or behavior.

**HumanMessage:** represents the user's message.

**AIMessage:** represents a previous assistant response.

Message-based prompting is useful because each part of the conversation keeps its role.
The model can therefore distinguish instructions, user questions and previous answers.

## Task 2 — MessagePlaceholder

`MessagesPlaceholder("chat_history")` creates a position in the prompt where the stored
conversation messages are inserted before the new user question.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}")
])

history = [
    HumanMessage(content="Explain Python lists."),
    AIMessage(content="A Python list stores multiple values in one variable.")
]

prompt_value = prompt.invoke({
    "chat_history": history,
    "question": "Give me an example."
})

print(prompt_value)


## Task 3 — Basic Conversation History

The history is stored as a Python list. After each turn, a HumanMessage and AIMessage
are added to the list.

In [ ]:
history = []

history.append(HumanMessage(content="Explain Python lists."))
history.append(AIMessage(content="A list stores multiple values in order."))

print("Messages stored:", len(history))
for message in history:
    print(type(message).__name__, ":", message.content)


## Task 4 — Trimming Chat History

Two limits are demonstrated:

1. Message-count limit: keep only the newest messages.
2. Approximate token limit: estimate tokens using roughly 4 characters per token.

The token method is intentionally an estimate because exact tokenization depends on the model.

In [ ]:
def trim_history(history, max_messages=6):
    if len(history) <= max_messages:
        return history
    return history[-max_messages:]


def trim_history_by_tokens(history, max_tokens=700):
    kept = []
    total = 0

    for message in reversed(history):
        estimated_tokens = max(1, len(str(message.content)) // 4)
        if total + estimated_tokens > max_tokens:
            break
        kept.insert(0, message)
        total += estimated_tokens

    return kept


print("Original:", len(history))
print("Message limit:", len(trim_history(history, 1)))
print("Token limit:", len(trim_history_by_tokens(history, 20)))


## Task 5 — Q&A Chatbot with Message History

Run `main.py` locally for the real Ollama responses. The required follow-up test is:

1. Explain Python lists in simple words.
2. Give me one example.
3. What about tuples?

The actual responses must be recorded in `MY_RUN_NOTES.txt`.

## Task 6 — Streamlit Chatbot

`app.py` uses Streamlit session state to keep messages while the browser session is active.
Run:

```text
streamlit run app.py
```

Then save the real conversation screenshot as `outputs/streamlit_chatbot.png`.

## Task 7 — Observations & Insights

### Why history matters
It allows follow-up questions to refer to earlier turns.

### Long memory vs performance
Longer history gives more context but increases input size, processing time and token usage.

### Summarize vs trim
Trim when only recent messages matter. Summarize when older information still matters.

### MessagesPlaceholder vs memory
The placeholder inserts stored messages into a prompt; it does not store them.
The application stores history separately.